In [1]:
import numpy as np

from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge, ElasticNet, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split


In [2]:
# ============================================================
# Load Data
# ============================================================
X = np.loadtxt("waves.csv", delimiter=",")
Y = np.loadtxt("enzymes.csv", delimiter=",")

if Y.ndim == 1:
    Y = Y.reshape(-1, 1)
n_targets = Y.shape[1]
n_features = X.shape[1]

In [3]:
n_samples, n_features = X.shape
n_targets = Y.shape[1]
print(f"X shape: {X.shape}, Y shape: {Y.shape}")

# Single consistent train/test split used everywhere
test_size = 0.2
random_state = 42

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=test_size, random_state=random_state
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train shape: {Y_train.shape}, Test shape: {Y_test.shape}")


X shape: (258, 1868), Y shape: (258, 5)
Train shape: (206, 1868), Test shape: (52, 1868)
Train shape: (206, 5), Test shape: (52, 5)


In [11]:
def evaluate_model(
    X, y,
    model_name,
    ridge_alpha=1.0,
    en_alpha=0.01,
    en_l1_ratio=0.5,
    lasso_alpha=1.0,
    max_iter=30000,
    max_pls_components=15,
    cv_splits=5,
    verbose=True,
    random_state=42
):
    """
    Train and evaluate model on (X, y).
    Scaling rules:
        - PLS, Ridge → StandardScaler
        - ElasticNet, Lasso → NO scaling
    PLS finds best n_components via CV.
    """

    rng = np.random.RandomState(random_state)

    # ============
    #     PLS
    # ============
    if model_name.lower() == "pls":
        max_comp = min(max_pls_components, X.shape[1], X.shape[0] - 1)
        best_score = -np.inf
        best_k = 1

        for k in range(1, max_comp + 1):
            pls = PLSRegression(n_components=k, scale=True)
            
            cv = KFold(n_splits=cv_splits, shuffle=True,
                    random_state=rng.randint(0, 1_000_000))
            
            score = cross_val_score(pls, X, y, cv=cv, scoring="r2").mean()
            
            if score > best_score:
                best_score = score
                best_k = k

        # Train final model
        model = PLSRegression(n_components=best_k, scale=True)
        model.fit(X, y)
        y_pred = model.predict(X).ravel()
        extra = f"(best n_components = {best_k})"

    # ============
    #     RIDGE
    # ============
    elif model_name.lower() == "ridge":
        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=ridge_alpha)
        )
        model.fit(X, y)
        y_pred = model.predict(X).ravel()
        extra = f"(alpha={ridge_alpha})"

    # ============
    #  ELASTIC NET
    # ============
    elif model_name.lower() == "elasticnet":
        model = ElasticNet(alpha=en_alpha, l1_ratio=en_l1_ratio, max_iter=max_iter)
        model.fit(X, y)
        y_pred = model.predict(X).ravel()
        extra = f"(alpha={en_alpha}, l1={en_l1_ratio})"

    # ============
    #     LASSO
    # ============
    elif model_name.lower() == "lasso":
        model = Lasso(alpha=lasso_alpha, max_iter=max_iter)
        model.fit(X, y)
        y_pred = model.predict(X).ravel()
        extra = f"(alpha={lasso_alpha})"

    else:
        raise ValueError("Invalid model_name")

    # ============
    #   METRICS
    # ============
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    rep = 100 * rmse / np.mean(np.abs(y))

    if verbose:
        print(f"  {model_name.upper()} {extra}")
        print(f"    R²   = {r2:.4f}")
        print(f"    RMSE = {rmse:.4f}")
        print(f"    REP% = {rep:.2f}")

    return model, {"r2": r2, "rmse": rmse, "rep": rep}


In [ ]:
def genetic_algorithm_feature_selection(
    X, 
    y, 
    n_generations=30,
    population_size=40,
    crossover_rate=0.8,
    mutation_rate=0.01,
    max_pls_components=20,
    cv_splits=5,
    random_state=42,
    verbose=True
):
    rng = np.random.RandomState(random_state)
    n_samples, n_features = X.shape
    
    # --- Helper: fitness for one individual ---
    def fitness(individual):
        cols = np.where(individual == 1)[0]
        if len(cols) == 0:
            return -9999

        pls = PLSRegression(n_components=8)
        pls.fit(X[:, cols], y)
        pred = pls.predict(X[:, cols]).ravel()
        r2 = r2_score(y, pred)
        return r2
    
    # --- Initialize population ---
    def init_population():
        # start with ~10% of features active in each individual
        pop = rng.rand(population_size, n_features) < 0.1
        # fix any all-zero individuals
        for i in range(population_size):
            if not pop[i].any():
                idx = rng.randint(0, n_features)
                pop[i, idx] = True
        return pop
    
    # --- Selection: tournament ---
    def tournament_selection(fitnesses, k=3):
        chosen_idx = rng.choice(len(fitnesses), size=k, replace=False)
        best_idx = chosen_idx[np.argmax(fitnesses[chosen_idx])]
        return best_idx
    
    # --- Crossover: one-point ---
    def crossover(parent1, parent2):
        if rng.rand() < crossover_rate:
            point = rng.randint(1, n_features - 1)
            child1 = np.concatenate([parent1[:point], parent2[point:]])
            child2 = np.concatenate([parent2[:point], parent1[point:]])
        else:
            child1, child2 = parent1.copy(), parent2.copy()
        return child1, child2
    
    # --- Mutation: bit flip ---
    def mutate(individual):
        mutation_mask = rng.rand(n_features) < mutation_rate
        individual[mutation_mask] = ~individual[mutation_mask]
        # avoid empty individual
        if not individual.any():
            idx = rng.randint(0, n_features)
            individual[idx] = True
        return individual
    
    # --- GA main loop ---
    population = init_population()
    fitnesses = np.array([fitness(ind) for ind in population])
    
    best_idx = np.argmax(fitnesses)
    best_individual = population[best_idx].copy()
    best_fitness = fitnesses[best_idx]
    
    if verbose:
        print(f"Initial best fitness (R²): {best_fitness:.4f}, "
              f"features: {best_individual.sum()}")
    
    for gen in range(1, n_generations + 1):
        new_population = []
        
        while len(new_population) < population_size:
            # select parents
            p1_idx = tournament_selection(fitnesses)
            p2_idx = tournament_selection(fitnesses)
            parent1 = population[p1_idx]
            parent2 = population[p2_idx]
            
            # crossover + mutation
            child1, child2 = crossover(parent1, parent2)
            child1 = mutate(child1)
            child2 = mutate(child2)
            
            new_population.extend([child1, child2])
        
        population = np.array(new_population[:population_size])
        fitnesses = np.array([fitness(ind) for ind in population])
        
        gen_best_idx = np.argmax(fitnesses)
        gen_best = population[gen_best_idx]
        gen_best_fit = fitnesses[gen_best_idx]
        
        if gen_best_fit > best_fitness:
            best_fitness = gen_best_fit
            best_individual = gen_best.copy()
        
        if verbose:
            print(f"Generation {gen:02d}: best R² = {gen_best_fit:.4f}, "
                  f"global best = {best_fitness:.4f}, "
                  f"features = {best_individual.sum()}")
    
    # --- After GA: evaluate different models on best feature subset ---
    selected_indices = np.where(best_individual)[0]
    X_sel = X[:, selected_indices]
    
    if verbose:
        print("\nFinal evaluation on training data with selected features "
              f"({len(selected_indices)} features):")
    
    model_candidates = ['pls', 'ridge', 'elasticnet', 'lasso']
    models_array = [] 

    best_model = None
    best_model_name = None
    best_metrics = None
    best_r2 = -np.inf
    
    for name in model_candidates:
        model, metrics = evaluate_model(
            X_sel, y, model_name=name,
            max_pls_components=max_pls_components,
            cv_splits=cv_splits,
            verbose=verbose,
            random_state=random_state
        )
        models_array.append(model)
        
        if metrics["r2"] > best_r2:
            best_r2 = metrics["r2"]
            best_model = model
            best_model_name = name
            best_metrics = metrics
    
    if verbose:
        print(f"\nBest model on training: {best_model_name.upper()} "
              f"with R² = {best_r2:.4f}")
    
    return selected_indices, best_model, best_model_name, models_array, best_metrics


In [6]:
from sklearn.cross_decomposition import PLSRegression

def is_pls(model):
    if hasattr(model, "steps"):
        return isinstance(model.steps[-1][1], PLSRegression)
    return False
from sklearn.linear_model import Ridge

def is_ridge(model):
    if hasattr(model, "steps"):
        return isinstance(model.steps[-1][1], Ridge)
    return False
from sklearn.linear_model import ElasticNet, Lasso

def is_elasticnet(model):
    return isinstance(model, ElasticNet)

def is_lasso(model):
    return isinstance(model, Lasso)


In [7]:
from sklearn.metrics import r2_score, mean_squared_error

def evaluate_on_validation(model, X_val_sel, y_val, verbose=True):
    """
    Evaluate an ALREADY TRAINED model on validation data.
    No retraining, no scaling here.
    Scaling and hyperparameters are already baked into `model`.
    """
    model.fit(X_val_sel, y_val)
    y_pred = model.predict(X_val_sel).ravel()

    r2 = r2_score(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mean_y = np.mean(np.abs(y_val))
    rep = 100.0 * rmse / mean_y if mean_y != 0 else np.nan

    if verbose:
        print(f"    Validation R²   = {r2:.4f}")
        print(f"    Validation RMSE = {rmse:.4f}")
        print(f"    Validation REP% = {rep:.2f}")

    return {"r2": r2, "rmse": rmse, "rep": rep}


In [8]:
def get_model_name(model):
    # Pipeline → model.steps list
    if hasattr(model, "steps"):
        final_step = model.steps[-1][1]
        return final_step.__class__.__name__
    else:
        return model.__class__.__name__


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
test_results = []
selected_features_per_enzyme = {}

n_enzymes = Y_train.shape[1]

for enzyme_idx in range(n_enzymes):
    print("=" * 70)
    print(f"Enzyme {enzyme_idx + 1}")
    
    y_train = Y_train[:, enzyme_idx]
    y_val = Y_test[:, enzyme_idx]
    
    # Run GA feature selection for this enzyme
    selected_features, trained_model, model_name, models_array, train_metrics = genetic_algorithm_feature_selection(
        X_train,
        y_train,
        n_generations=40,        
        population_size=50,
        crossover_rate=0.8,
        mutation_rate=0.01,
        max_pls_components=15,   
        cv_splits=5,
        random_state=42 + enzyme_idx,
        verbose=True
    )
    
    print(f"\nEnzyme {enzyme_idx + 1}:")
    print(f"  Selected {len(selected_features)} features.")
    # print(f"  Selected feature indices: {selected_features}")
    print(f"  Best training model: {model_name.upper()}")
    
    # Store selected features for this enzyme
    selected_features_per_enzyme[enzyme_idx + 1] = selected_features
    
    # Evaluate on validation data using same selected features
    X_val_sel = X_test[:, selected_features]
    # print("  Validation performance for ALL models:")

    for model in models_array:
        model_name = get_model_name(model)

        val_metrics = evaluate_on_validation(
            model,
            X_val_sel=X_test[:, selected_features],
            y_val=y_val,
            verbose=False
        )

        test_results.append({
            "Enzyme": enzyme_idx + 1,
            "Feature_Selection": "GA",   
            "Model": model_name,
            "R2": (val_metrics["r2"]),
            "RMSE": (val_metrics["rmse"]),
            "REP%": (val_metrics["rep"]),
            "N_Features": len(selected_features)
        })

results_df = pd.DataFrame(test_results)

# Sort for readability
results_df = results_df.sort_values(
    by=["Feature_Selection", "Enzyme", "Model"]
)

print("\nFINAL TEST RESULTS TABLE:")
print(results_df.to_string(index=False))

# Print summary of selected features for all enzymes
print("\n" + "=" * 70)
print("SELECTED FEATURES SUMMARY:")
print("=" * 70)
for enzyme_num, features in selected_features_per_enzyme.items():
    print(f"\nEnzyme {enzyme_num}:")
    print(f"  Number of selected features: {len(features)}")
    print(f"  Feature indices: {features}")
    print(f"  Feature indices (formatted): {list(features)}")


Enzyme 1
Initial best fitness (R²): 0.7072, features: 204
Generation 01: best R² = 0.7118, global best = 0.7118, features = 221
Generation 02: best R² = 0.7376, global best = 0.7376, features = 236
Generation 03: best R² = 0.7413, global best = 0.7413, features = 252
Generation 04: best R² = 0.7510, global best = 0.7510, features = 272
Generation 05: best R² = 0.7566, global best = 0.7566, features = 278
Generation 06: best R² = 0.7685, global best = 0.7685, features = 293
Generation 07: best R² = 0.7768, global best = 0.7768, features = 313
Generation 08: best R² = 0.7818, global best = 0.7818, features = 325
Generation 09: best R² = 0.7808, global best = 0.7818, features = 325
Generation 10: best R² = 0.7884, global best = 0.7884, features = 349
Generation 11: best R² = 0.7939, global best = 0.7939, features = 375
Generation 12: best R² = 0.7928, global best = 0.7939, features = 375
Generation 13: best R² = 0.7940, global best = 0.7940, features = 400
Generation 14: best R² = 0.8018,

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.558e+07, tolerance: 7.164e+04
  model = cd_fast.enet_coordinate_descent(


  ELASTICNET (alpha=0.01, l1=0.5)
    R²   = 0.8069
    RMSE = 819.4255
    REP% = 36.05
  LASSO (alpha=1.0)
    R²   = 0.9255
    RMSE = 508.8823
    REP% = 22.39

Best model on training: RIDGE with R² = 0.9981

Enzyme 1:
  Selected 609 features.
  Best training model: RIDGE


c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.628e+05, tolerance: 1.964e+04
  model = cd_fast.enet_coordinate_descent(


Enzyme 2
Initial best fitness (R²): 0.6852, features: 184
Generation 01: best R² = 0.7044, global best = 0.7044, features = 194
Generation 02: best R² = 0.7162, global best = 0.7162, features = 230
Generation 03: best R² = 0.7247, global best = 0.7247, features = 242
Generation 04: best R² = 0.7425, global best = 0.7425, features = 265
Generation 05: best R² = 0.7492, global best = 0.7492, features = 261
Generation 06: best R² = 0.7697, global best = 0.7697, features = 291
Generation 07: best R² = 0.7650, global best = 0.7697, features = 291
Generation 08: best R² = 0.7715, global best = 0.7715, features = 314
Generation 09: best R² = 0.7766, global best = 0.7766, features = 334
Generation 10: best R² = 0.7792, global best = 0.7792, features = 342
Generation 11: best R² = 0.7834, global best = 0.7834, features = 353
Generation 12: best R² = 0.7854, global best = 0.7854, features = 365
Generation 13: best R² = 0.7877, global best = 0.7877, features = 384
Generation 14: best R² = 0.7909,

In [13]:
results_df.to_csv("GA test_results_summary.csv", index=False)